<a href="https://colab.research.google.com/github/PrashVenga/AMLDM-CW-PART-1-FINAL-SUB-1-/blob/IBM-CW1-Part-1-Final-Sub-Codes/IBM_CW2_Part3_(Final)_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Part 3 Association Rule Learning Analysis (TV Shows)**

This notebook requires the dataset:

TV Shows – Association Rule Learning.csv

# Before running the notebook:

Upload the dataset file to your Google Drive.

Place it inside your main MyDrive directory (or update the file path accordingly).

Ensure the file name matches exactly:

TV Shows - Association Rule Learning.csv

# **Imports**

The required libraries were imported to perform association rule mining. mlxtend provides the Apriori algorithm and rule generation functions, while pandas and numpy are used for data manipulation and preprocessing.

In [25]:
# Imports
!pip install mlxtend
import mlxtend
import pandas as pd
import numpy as np

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# **Mounting Google Drive**

Google Drive was mounted to enable access to the TV Shows dataset stored in the Drive directory for analysis in Google Colab.

In [26]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# **Dataset Loading**

The dataset was loaded from Google Drive using pandas. The parameter header=None was specified to ensure that all rows were treated as transactional data rather than column headers. The first few rows were displayed to verify that the dataset was loaded correctly.

In [27]:
file_path = "/content/drive/MyDrive/TV Shows - Association Rule Learning.csv" # replace file if needed

df = pd.read_csv(file_path, header=None)
df.head()

,0,1,2,3,4,5,6,7,8,9,...,22,23,24,25,26,27,28,29,30,31
0,Cobra Kai,Lupin,12 Monkeys,Sherlock,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Lost,Jack Ryan,The Flash,Game of thrones,House of Cards,12 Monkeys,Vikings,Fringe,The Mentalist,The Alienist,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Sex Education,Dr. House,Kingdom,The Walking Dead,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Ozark,Sex Education,Constantine,Preacher,Vikings,The Tick,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Naruto,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# **Transaction Preparation**

Each row of the dataset was converted into a list to represent an individual user transaction. Missing values (NaN) were removed to ensure that each transaction contained only valid TV show titles. The first few transactions were displayed to confirm correct preprocessing.

In [28]:
# Converting dataframe rows into list format
transactions = df.values.tolist()

# Removing NaN values from each transaction
transactions = [
    [item for item in transaction if pd.notna(item)]
    for transaction in transactions
]

# Previewing first 5 transactions
transactions[:5]

[['Cobra Kai', 'Lupin', '12 Monkeys', 'Sherlock'],
 ['Lost',
  'Jack Ryan',
  'The Flash',
  'Game of thrones',
  'House of Cards',
  '12 Monkeys',
  'Vikings',
  'Fringe',
  'The Mentalist',
  'The Alienist',
  'Big Little Lies',
  'Chernobyl'],
 ['Sex Education', 'Dr. House', 'Kingdom', 'The Walking Dead'],
 ['Ozark', 'Sex Education', 'Constantine', 'Preacher', 'Vikings', 'The Tick'],
 ['Naruto']]

# **Transaction Encoding**

The transactions were transformed into a one-hot encoded format using TransactionEncoder. In this representation, each column corresponds to a TV show and each row indicates whether a user watched that show (True/False). This encoded matrix is required for applying the Apriori algorithm.

In [29]:
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)

df_encoded = pd.DataFrame(te_array, columns=te.columns_)

df_encoded.head()

,12 Monkeys,24,Absentia,Alice in Borderland,Altered Carbon,American Gods,Another Life,Archer,Arrow,Atypical,...,True Detective,Two and a half men,Upload,Vikings,Watchmen,Westworld,White Collar,X-Files,You,Young Sheldon
0,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,True,False,False,False,False,False,False,False,False,False,...,False,False,False,True,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,True,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


# **Frequent Itemset Generation (Apriori)**

The Apriori algorithm was applied to the encoded dataset with a minimum support threshold of 5%. This step identifies individual TV shows or combinations of shows that appear in at least 5% of user transactions, allowing frequently co-watched patterns to be extracted.

In [30]:
frequent_itemsets = apriori(
    df_encoded,
    min_support=0.05,
    use_colnames=True
)

# **Association Rule Generation**

Association rules were generated from the frequent itemsets using lift as the evaluation metric. A minimum lift threshold of 1.0 was applied to retain only positively associated show combinations. The rules were sorted in descending order of lift to identify the strongest co-viewing relationships.

In [31]:
rules = association_rules(
    frequent_itemsets,
    metric="lift",
    min_threshold=1.0
)

rules.sort_values(by="lift", ascending=False).head(10)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
1,(Atypical),(Sex Education),0.139938,0.255624,0.056140,0.401180,1.569412,1.0,0.020369,1.243071,0.421852,0.165400,0.195541,0.310400
0,(Sex Education),(Atypical),0.255624,0.139938,0.056140,0.219621,1.569412,1.0,0.020369,1.102107,0.487413,0.165400,0.092647,0.310400
2,(Sex Education),(Ozark),0.255624,0.193705,0.075129,0.293904,1.517277,1.0,0.025613,1.141906,0.458001,0.200772,0.124271,0.340878
3,(Ozark),(Sex Education),0.193705,0.255624,0.075129,0.387853,1.517277,1.0,0.025613,1.216008,0.422828,0.200772,0.177637,0.340878
4,(Sex Education),(Two and a half men),0.255624,0.183591,0.056553,0.221235,1.205043,1.0,0.009623,1.048338,0.228586,0.147789,0.046109,0.264637
5,(Two and a half men),(Sex Education),0.183591,0.255624,0.056553,0.308038,1.205043,1.0,0.009623,1.075747,0.208417,0.147789,0.070413,0.264637
